# 📦 Phase 1: Data Preparation
Load, clean, engineer features, and save processed datasets.

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# ── Locate CSV automatically ──────────────────────────────────────────────────
search_names = [
    'student_placement_career_success_dataset.csv',
    'student_placement.csv',
    'dataset.csv'
]
search_dirs = ['.', '..', 'data', os.path.expanduser('~'), os.path.expanduser('~/Downloads')]

csv_path = None
for d in search_dirs:
    for n in search_names:
        p = os.path.join(d, n)
        if os.path.exists(p):
            csv_path = p
            break
    if csv_path: break

if csv_path is None:
    raise FileNotFoundError(
        "❌ CSV not found! Copy your dataset CSV into this project folder "
        "and rename it: student_placement_career_success_dataset.csv"
    )

print(f"✅ Dataset found: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
df.head(3)


## 🔍 Basic Info & Data Types

In [ ]:
print("=== Dataset Info ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n=== Duplicates ===", df.duplicated().sum())


## 🧹 Handle Missing Values

In [ ]:
df.drop_duplicates(inplace=True)

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

for c in num_cols:
    df[c].fillna(df[c].median(), inplace=True)
for c in cat_cols:
    df[c].fillna(df[c].mode()[0], inplace=True)

print("✅ Missing values handled. Shape:", df.shape)


## 🏗️ Feature Engineering — 4 New Columns

In [ ]:
# 1. Coding Score
df['coding_score'] = (
    df['DSA_problems_solved'] * 0.5 +
    df['GitHub_repos'] * 10 +
    df['hackathons_participated'] * 20 +
    df['development_projects_count'] * 15 +
    df['AI_ML_projects'] * 25
) / 100

# 2. AI Readiness Score  
df['ai_readiness_score'] = (
    df['AI_tool_usage_frequency'].map({'Never':0,'Rarely':1,'Sometimes':2,'Often':3,'Daily':4}).fillna(0) * 20 +
    df['prompt_engineering_skill'] * 10 +
    (10 - df['AI_fear_score']) * 5 +
    df['adaptability_score'] * 8
) / 100

# 3. Wellness Score
df['wellness_score'] = (
    df['sleep_hours'] * 10 +
    (24 - df['screen_time']) * 5 +
    (8 - df['gaming_hours'].clip(0,8)) * 5 +
    (10 - df['stress_level']) * 8 +
    df['gym_frequency'] * 6
) / 100

# 4. Interview Readiness
df['interview_readiness'] = (
    df['mock_interview_score'] * 0.3 +
    df['communication_skills'] * 0.25 +
    df['aptitude_score'] * 0.25 +
    df['resume_score'] * 0.2
)

print("✅ Feature engineering done!")
print(df[['coding_score','ai_readiness_score','wellness_score','interview_readiness']].describe())


## 🔤 Encoding & Scaling

In [ ]:
df_processed = df.copy()
le = LabelEncoder()

for c in cat_cols:
    if c != 'placement_status':
        df_processed[c] = le.fit_transform(df_processed[c].astype(str))

# Encode target
df_processed['placement_status_enc'] = (df_processed['placement_status'] == 'Placed').astype(int)

scaler = StandardScaler()
scale_cols = [c for c in num_cols if c not in ['student_id']]
df_processed[scale_cols] = scaler.fit_transform(df_processed[scale_cols])

print("✅ Encoding & scaling done!")


## 💾 Save Processed Datasets

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Full processed data
df_processed.to_csv('data/processed_data.csv', index=False)

# ── Placement dataset ─────────────────────────────────────────────────────────
placement_features = [
    'cgpa','backlog_history','attendance_percentage','DSA_problems_solved',
    'GitHub_repos','hackathons_participated','development_projects_count',
    'AI_ML_projects','internships_completed','resume_score','communication_skills',
    'aptitude_score','mock_interview_score','college_tier','branch',
    'coding_score','ai_readiness_score','interview_readiness'
]
placement_features = [c for c in placement_features if c in df_processed.columns]
X_place = df_processed[placement_features]
y_place = df_processed['placement_status_enc']
X_tr, X_te, y_tr, y_te = train_test_split(X_place, y_place, test_size=0.2, random_state=42, stratify=y_place)
X_tr.to_csv('data/X_train_placement.csv', index=False)
X_te.to_csv('data/X_test_placement.csv', index=False)
y_tr.to_csv('data/y_train_placement.csv', index=False)
y_te.to_csv('data/y_test_placement.csv', index=False)

# ── Salary dataset (placed students only) ────────────────────────────────────
df_placed = df_processed[df_processed['placement_status_enc'] == 1].copy()
salary_features = [
    'cgpa','college_tier','branch','coding_score','ai_readiness_score',
    'interview_readiness','internships_completed','offer_count',
    'interview_rounds_cleared','company_type'
]
salary_features = [c for c in salary_features if c in df_placed.columns]
X_sal = df_placed[salary_features]
y_sal = df_placed['salary_lpa']
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_sal, y_sal, test_size=0.2, random_state=42)
Xs_tr.to_csv('data/X_train_salary.csv', index=False)
Xs_te.to_csv('data/X_test_salary.csv', index=False)
ys_tr.to_csv('data/y_train_salary.csv', index=False)
ys_te.to_csv('data/y_test_salary.csv', index=False)

# ── Burnout dataset ───────────────────────────────────────────────────────────
burnout_features = [
    'sleep_hours','screen_time','gaming_hours','study_hours_daily',
    'stress_level','gym_frequency','self_learning_hours','motivation_level',
    'wellness_score','cgpa','attendance_percentage'
]
burnout_features = [c for c in burnout_features if c in df_processed.columns]
X_burn = df_processed[burnout_features]
y_burn = (df_processed['burnout_score'] > df_processed['burnout_score'].median()).astype(int)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_burn, y_burn, test_size=0.2, random_state=42, stratify=y_burn)
Xb_tr.to_csv('data/X_train_burnout.csv', index=False)
Xb_te.to_csv('data/X_test_burnout.csv', index=False)
yb_tr.to_csv('data/y_train_burnout.csv', index=False)
yb_te.to_csv('data/y_test_burnout.csv', index=False)

print("✅ All 12 dataset files saved in data/ folder!")
print(f"  Placement train: {X_tr.shape}, test: {X_te.shape}")
print(f"  Salary train:    {Xs_tr.shape}, test: {Xs_te.shape}")
print(f"  Burnout train:   {Xb_tr.shape}, test: {Xb_te.shape}")
